# Visual-language assistant with Ministral-3 and OpenVINO

Ministral-3 is a family of compact multimodal models from [Mistral AI](https://mistral.ai/) that combines a language model with a Pixtral vision encoder. This tutorial supports instruction-following checkpoints and the reasoning-post-trained `Ministral-3-3B-Reasoning-2512` model.

**Key Features of Ministral-3:**

* **Multimodal Understanding** — Processes images and text for visual question answering and image understanding.
* **Reasoning** — The Reasoning checkpoint produces a `[THINK]...[/THINK]` trace before the final answer and benefits from multi-turn preservation of that trace.
* **Long Context Support** — Supports up to 262,144 tokens with YaRN RoPE scaling.
* **Efficient Architecture** — The 3B variant combines a 3.4B language model with a 0.4B Pixtral vision encoder.

See the [Instruct model card](https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512-BF16), the [Reasoning model card](https://huggingface.co/mistralai/Ministral-3-3B-Reasoning-2512), and the [Mistral AI documentation](https://docs.mistral.ai/) for more details.

In this tutorial, we convert and optimize Ministral-3 with [Optimum Intel](https://github.com/huggingface/optimum-intel) and [NNCF](https://github.com/openvinotoolkit/nncf), then run image-text generation with `OVModelForVisualCausalLM`. For the Reasoning checkpoint, the notebook also separates the reasoning trace from the final answer and preserves it in chat history.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert and Optimize model](#Convert-and-Optimize-model)
    - [Select weight format](#Select-weight-format)
- [Prepare OpenVINO Inference Pipeline](#Prepare-OpenVINO-Inference-Pipeline)
    - [Select inference device](#Select-inference-device)
    - [Load OpenVINO model](#Load-OpenVINO-model)
- [Run OpenVINO model inference](#Run-OpenVINO-model-inference)
- [Interactive Demo](#Interactive-Demo)

⚠️ **EXPERIMENTAL NOTEBOOK**

Mistral3 support was recently added to Optimum Intel. The Reasoning checkpoint and its INT4-compressed variant should be validated end to end before this tutorial is considered stable. Video, audio, and NPU inference are not supported in this example.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/ministral-3/ministral-3.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to the [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Install required packages and setup helper functions.

In [ ]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q "transformers==5.0.0" "huggingface_hub==1.14.0" "nncf==3.1.0" "torch==2.8" "torchvision==0.23.0" "peft>=0.15.0" "Pillow" "gradio>=4.36,<6" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "openvino>=2026.0.0" "openvino-tokenizers>=2026.0.0"
# Optimum Intel commit from merged PR #1627 adds Mistral3 export and inference support.
%pip install -q --upgrade-strategy eager "optimum-intel[openvino,nncf] @ git+https://github.com/huggingface/optimum-intel.git@b254b32cd9c18ad3bfa6fbe41267cf3b6fabba38" --extra-index-url https://download.pytorch.org/whl/cpu

In [2]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("ministral-3.ipynb")

## Select model
[back to top ⬆️](#Table-of-contents:)

Select an instruction-following model or the 3B Reasoning checkpoint. The Reasoning variant uses the same Mistral3 architecture and conversion path, but requires reasoning-aware generation and output handling.

In [ ]:
import ipywidgets as widgets

model_ids = [
    "mistralai/Ministral-3-3B-Instruct-2512-BF16",
    "mistralai/Ministral-3-3B-Reasoning-2512",
    "mistralai/Ministral-3-8B-Instruct-2512-BF16",
]

model_id = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
)

model_id

In [ ]:
print(f"Selected: {model_id.value}")
pt_model_id = model_id.value
model_dir = Path(pt_model_id.split("/")[-1])
is_reasoning_model = "-Reasoning-" in pt_model_id
print(f"Reasoning mode: {is_reasoning_model}")

## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

Ministral-3 is a PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). For convenience, we will use OpenVINO integration with HuggingFace Optimum.

🤗 [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the 🤗 Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

`optimum-cli` provides command line interface for model conversion and optimization.

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where `task` is task to export the model for. Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`. For int8 and int4, [NNCF](https://github.com/openvinotoolkit/nncf) will be used for weight compression.

### Select weight format
[back to top ⬆️](#Table-of-contents:)

For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf) and fixed-precision quantization. Weights compression reduces the memory footprint of the model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs).

In [6]:
import ipywidgets as widgets

to_compress = widgets.Checkbox(
    value=True,
    description="INT4 weight compression",
    disabled=False,
)

to_compress

Checkbox(value=True, description='INT4 weight compression')

In [7]:
additional_args = {"task": "image-text-to-text"}

if to_compress.value:
    model_export_dir = model_dir / "INT4"
    additional_args.update({"weight-format": "int4"})
else:
    model_export_dir = model_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})

print(f"Model will be exported to: {model_export_dir}")

from cmd_helper import optimum_cli

if not model_export_dir.exists():
    optimum_cli(pt_model_id, model_export_dir, additional_args=additional_args)

Model will be exported to: Ministral-3-3B-Instruct-2512-BF16/INT4


## Prepare OpenVINO Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

OpenVINO integration with Optimum Intel provides ready-to-use API for model inference that can be used for smooth integration with transformers-based solutions. For loading model, we will use `OVModelForVisualCausalLM` class that has compatible API with Transformers models and provides the following interface for interaction:

* `from_pretrained` - for loading model from directory.
* `generate` - for running model inference.
* `preprocess_inputs` - for preparing model inputs.

In [ ]:
from optimum.intel.openvino import OVModelForVisualCausalLM
from transformers import AutoProcessor

### Select inference device
[back to top ⬆️](#Table-of-contents:)

In [9]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])

device

Dropdown(description='Device:', index=3, options=('CPU', 'GPU.0', 'GPU.1', 'AUTO'), value='AUTO')

### Load OpenVINO model
[back to top ⬆️](#Table-of-contents:)

For model loading we should provide path to model directory and inference device.

In [10]:
model_export_dir = model_dir / ("INT4" if to_compress.value else "FP16")

# Enable OpenVINO model cache to avoid recompilation after kernel restart
ov_config = {"CACHE_DIR": str(model_export_dir / ".ov_cache")}

model = OVModelForVisualCausalLM.from_pretrained(model_export_dir, device=device.value, ov_config=ov_config)
processor = AutoProcessor.from_pretrained(model_export_dir)

print(f"Model loaded on {device.value}")

Model loaded on AUTO


## Run OpenVINO model inference
[back to top ⬆️](#Table-of-contents:)

Now that the model and processor are loaded, we can run image-text inference. The `preprocess_inputs` method applies the model's chat template, including the default system prompt recommended for the Reasoning checkpoint.

The Reasoning model is sampled with the model-card defaults (`temperature=0.7`, `top_p=0.95`) and a larger token budget. Its `[THINK]...[/THINK]` section is displayed separately from the final answer.

In [11]:
from PIL import Image
from io import BytesIO

MAX_IMAGE_SIZE = 512


def load_image(image_file):
    if isinstance(image_file, str) and (image_file.startswith("http") or image_file.startswith("https")):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_file).convert("RGB")
    # Resize large images to keep patch count manageable
    if max(image.size) > MAX_IMAGE_SIZE:
        image.thumbnail((MAX_IMAGE_SIZE, MAX_IMAGE_SIZE))
    return image


image_url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
image_file = Path("demo.jpeg")
text_message = "Describe this image."

if not image_file.exists():
    image = load_image(image_url)
    image.save(image_file)
else:
    image = load_image(image_file)

inputs = model.preprocess_inputs(text=text_message, image=image, processor=processor)

In [ ]:
def strip_terminal_tokens(text, tokenizer):
    for token in (tokenizer.bos_token, tokenizer.eos_token, tokenizer.pad_token):
        if token:
            text = text.replace(token, "")
    return text.strip()


def split_reasoning_output(text, tokenizer):
    think_start = "[THINK]"
    think_end = "[/THINK]"
    end_pos = text.find(think_end)
    if end_pos < 0:
        return None, strip_terminal_tokens(text.replace(think_start, ""), tokenizer)

    start_pos = text.find(think_start)
    reasoning_start = start_pos + len(think_start) if start_pos >= 0 else 0
    reasoning = strip_terminal_tokens(text[reasoning_start:end_pos], tokenizer)
    answer = strip_terminal_tokens(text[end_pos + len(think_end) :], tokenizer)
    return reasoning, answer


generation_kwargs = (
    {"do_sample": True, "temperature": 0.7, "top_p": 0.95, "max_new_tokens": 1024}
    if is_reasoning_model
    else {"do_sample": False, "max_new_tokens": 128}
)
output_ids = model.generate(**inputs, **generation_kwargs)
prompt_length = inputs["input_ids"].shape[-1]
generated_text = processor.tokenizer.decode(output_ids[0, prompt_length:], skip_special_tokens=not is_reasoning_model)
reasoning, answer = split_reasoning_output(generated_text, processor.tokenizer)

print(f"Question:\n{text_message}")
display(image)
if reasoning:
    print(f"Reasoning:\n{reasoning}\n")
print(f"Answer:\n{answer}")

## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Try a multi-turn conversation with the model. Upload one or more images, enter a question, and submit it. For the Reasoning checkpoint, the demo displays `[THINK]...[/THINK]` content in a collapsible section and keeps the structured reasoning trace in subsequent model context.

In [ ]:
if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/ministral-3/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(model, processor, is_reasoning_model=is_reasoning_model)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/